# Rasha sam pa sam sa tha pa khunaya raha sam pa sam

In [20]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.naive_bayes import MultinomialNB
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, f1_score
from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.multiclass import  OneVsRestClassifier

In [21]:
df = pd.read_csv("../dataset/dataset_13_labels.csv")

In [22]:
df["text"] = df["title"].fillna("") + " " + df["body"].fillna("")
df.drop(["title", "body"], axis=1, inplace=True)

### Transform Labels (Multi-Hot Encoding)

In [23]:
import ast
df['labels-mapped'] = df['labels-mapped'].apply(lambda x: ast.literal_eval(x) if isinstance(x, str) else x)

mlb = MultiLabelBinarizer()
y = mlb.fit_transform(df["labels-mapped"])

print(mlb.classes_)

['bug' 'build_ci_cd' 'compatibility' 'documentation' 'feature_request'
 'integration_api' 'maintenance_data' 'performance' 'question'
 'security_access' 'setup_config' 'testing' 'ui_ux']


In [24]:
X_train, X_test, y_train, y_test = train_test_split(df['text'], y, test_size = 0.2, random_state = 42)

models = {
    "logreg": OneVsRestClassifier(LogisticRegression(max_iter=3000, class_weight="balanced")),
    "svm": OneVsRestClassifier(LinearSVC(class_weight="balanced"))
}

results = []

for name, clf in models.items():
    pipe = Pipeline([
        ("tdif", TfidfVectorizer(
        lowercase=True,
        stop_words="english",
        ngram_range=(1,2),
        min_df=2, 
        max_features=50000,
        sublinear_tf=True)),

        ("clf", clf)
    ])

    pipe.fit(X_train, y_train)
    y_pred = pipe.predict(X_test)

    acc_score = accuracy_score(y_test, y_pred)
    f1_val = f1_score(y_test, y_pred, average='macro')

    results.append((name, acc_score, f1_val))
    print(f"\n{name}")
    print("acc_scoreuracy:", acc_score)
    print("macro f1_val:", f1_val)
    from sklearn.metrics import classification_report
    print(classification_report(y_test, y_pred, target_names=mlb.classes_))



logreg
acc_scoreuracy: 0.24815265176316528
macro f1_val: 0.5950530840066939
                  precision    recall  f1-score   support

             bug       0.77      0.83      0.80      8547
     build_ci_cd       0.66      0.89      0.76      3457
   compatibility       0.53      0.80      0.64      3576
   documentation       0.38      0.73      0.50      1390
 feature_request       0.71      0.82      0.76      6273
 integration_api       0.43      0.75      0.55      1996
maintenance_data       0.28      0.68      0.40      1063
     performance       0.55      0.83      0.66      2261
        question       0.45      0.73      0.56      3107
 security_access       0.29      0.78      0.42       322
    setup_config       0.25      0.74      0.37       523
         testing       0.41      0.79      0.54      1732
           ui_ux       0.70      0.86      0.77      5671

       micro avg       0.58      0.81      0.67     39918
       macro avg       0.49      0.79      0.60    

c:\Users\arsal\Desktop\support-ticket-triage-system\venv\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in samples with no predicted labels. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])



svm
acc_scoreuracy: 0.30053315873164343
macro f1_val: 0.6154280203435605
                  precision    recall  f1-score   support

             bug       0.76      0.81      0.78      8547
     build_ci_cd       0.70      0.84      0.77      3457
   compatibility       0.55      0.74      0.63      3576
   documentation       0.45      0.64      0.53      1390
 feature_request       0.71      0.78      0.74      6273
 integration_api       0.48      0.65      0.55      1996
maintenance_data       0.37      0.54      0.44      1063
     performance       0.62      0.77      0.69      2261
        question       0.46      0.67      0.54      3107
 security_access       0.44      0.68      0.54       322
    setup_config       0.38      0.56      0.45       523
         testing       0.48      0.68      0.56      1732
           ui_ux       0.71      0.83      0.77      5671

       micro avg       0.62      0.76      0.68     39918
       macro avg       0.55      0.71      0.62     39

c:\Users\arsal\Desktop\support-ticket-triage-system\venv\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in samples with no predicted labels. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
